# DRC — Run the whole pipeline (one notebook, resumes across sessions)

Every stage end to end (data, training, evaluation, analysis, figures) in **one notebook** — no Add Input, no dataset chaining. The trick for going past the 12 h cap on a single T4: turn on **Persistence** (Settings -> Persistence -> *Files only*). That keeps `/kaggle/working` between sessions of THIS notebook, so when a session ends mid-training you just start a new one and **re-run** — finished models and stages are detected and skipped, and it continues until done. No files to move yourself. (The `01a`/`01b` split is the alternative if you'd rather not use Persistence.)

**This notebook is resumable.** Every stage runs in its own subprocess via
`drc.pipeline.run_pipeline`, so a CUDA OOM or a hard kernel crash in one stage
takes down that subprocess and *nothing else* — the kernel survives, the
failure is recorded, and the rest of the pipeline proceeds where it can. If
Kaggle's 12-hour session limit cuts you off, just **re-run the notebook**:
finished stages detect their own outputs and skip, so you pick up where you
stopped instead of redoing hours of work.

**Scope:** every stage (`only=None`). Set `FORCE_SINGLE_GPU = True` for one T4.

## Before you run

- Set the accelerator to **GPU T4 x2** (Settings -> Accelerator -> *GPU T4 x2*).
  With two cards the training sweep runs two jobs in parallel; with one it falls
  back to a 48-run single-GPU plan.
- **Precision is automatic.** The trainer picks bf16 on GPUs that support it,
  fp16 (with a GradScaler) on a T4, and fp32 on CPU — so you don't need to edit
  `configs/base.yaml`, even though it says `bf16`.

## The notebooks (split for a single-GPU, two-session run)

A single-T4 run is ~13-14 h, over Kaggle's 12 h cap, so the work is split so each
half fits one session:

1. **`kaggle_00_smoke`** — the whole pipeline on a tiny config (~10 min). Run it
   first to confirm every stage works before the long run.
2. **`kaggle_01a_data_train`** — session 1 (the long pole): download, parse,
   audit, dose corpora, tokenizer, and the 63-model training sweep. Ends with a
   health check on the trained models. If the 12 h cap cuts it off, just re-run —
   finished models are skipped, so it resumes.
3. **`kaggle_01b_eval_analysis`** — session 2 (~2-3 h): SLOR + n-gram evaluation,
   then all the analysis (Hill fits, E0 index, transfer, predictability,
   generalization, decision) and figures.

They chain through Kaggle's **notebook-output datasets**: run `01a` to completion,
*Save Version*, then in `01b` add that output as an input (*Add Input -> Your
Datasets*). The **restore-prior-artifacts** cell copies its `data/`, `models/`,
and `results/` into the working repo so `01b` finds the trained models.
(`kaggle_run_all` runs everything in one go, for reference / dual-GPU.)

Nothing here fabricates results. A blocked or failed stage produces no numbers —
it just says so in the dashboard and lets the rest proceed.

In [ ]:
# --- Setup: make the `drc` package importable, idempotently. -------------------
# Safe to re-run. If `drc` already imports we do nothing heavy. Otherwise we find
# the repo (uploaded as a dataset, or already cloned) or clone it, then install.
import os, sys, glob, subprocess
from pathlib import Path

# Clone URL, used only if the repo isn't already present (as a Kaggle dataset or
# a prior /kaggle/working checkout). HTTPS clones a PUBLIC repo with no auth; for
# a private repo, upload it as a Kaggle dataset instead (the cell finds it under
# /kaggle/input), or put a token in the URL: https://<token>@github.com/owner/repo.git
REPO_URL = "https://github.com/keshavkrishnan08/EMNLP.git"
WORK = Path("/kaggle/working/EMNLP")

# Master notebook: full stack; deps installed next cell.
EXTRA = "train"               # package extra to try, or "" for none
PIP_PACKAGES = ""  # plain pip fallbacks this notebook needs


def _have_drc() -> bool:
    try:
        import drc  # noqa: F401
        return True
    except Exception:
        return False


def _find_repo() -> Path | None:
    """Look for an existing checkout: a uploaded dataset or a prior /kaggle/working."""
    candidates = []
    # A repo uploaded as a Kaggle dataset shows up under /kaggle/input/<name>/...
    # Match it however it's nested: a `src/drc` at one or two levels down, or a
    # named folder. We don't assume any particular dataset/repo name.
    candidates += glob.glob("/kaggle/input/*/src/drc")
    candidates += glob.glob("/kaggle/input/*/*/src/drc")
    candidates += glob.glob("/kaggle/input/*/EMNLP")
    candidates += glob.glob("/kaggle/input/*")
    candidates += [str(WORK)]
    for c in candidates:
        c = Path(c)
        # Normalise: we want the repo ROOT (the dir that contains src/drc).
        root = c
        if root.name == "drc" and root.parent.name == "src":
            root = root.parent.parent
        if (root / "src" / "drc").exists() or (root / "pyproject.toml").exists():
            return root
    return None


def _run(cmd: str) -> int:
    print("$", cmd)
    return subprocess.call(cmd, shell=True)


# Locate the repo: an existing checkout, an uploaded dataset, or a fresh clone.
# We do this even if `drc` already imports, because a stale checkout from an
# earlier run would otherwise pin you to old code (and silently re-run old bugs).
repo = _find_repo()
if repo is None:
    print(f"No local repo found; cloning {REPO_URL} -> {WORK}")
    WORK.parent.mkdir(parents=True, exist_ok=True)
    _run(f'git clone --depth 1 "{REPO_URL}" "{WORK}"')
else:
    WORK = repo
    print(f"Using repo at {WORK}")
    # If it's a writable git checkout, pull the latest so code fixes actually land.
    # This is the difference between re-running old buggy code and the current fix.
    # (Tracked files only — your gitignored data/ models/ results/ are untouched.)
    if (WORK / ".git").exists():
        rc = subprocess.call(
            f'git -C "{WORK}" fetch --depth 1 origin main '
            f'&& git -C "{WORK}" reset --hard origin/main', shell=True,
        )
        print("Updated checkout to latest origin/main."
              if rc == 0 else "[warn] could not update checkout; using it as-is.")

# Install — ALWAYS, whether we just cloned or found an existing checkout. (This
# block is intentionally at top level, not under the else above: a fresh clone
# must still install its deps.) Editable install registers `drc` and pulls deps;
# note the extras brackets go INSIDE the quotes — pip install -e "PATH[extra]" —
# or the shell splits off "[extra]" and pip errors.
installed = False
if EXTRA:
    installed = subprocess.call(f'pip install -q -e "{WORK}[{EXTRA}]"', shell=True) == 0
    if not installed:
        print("[warn] editable install with the extra failed; trying plain -e")
if not installed:
    installed = subprocess.call(f'pip install -q -e "{WORK}"', shell=True) == 0
if not installed:
    print("[warn] editable install failed; relying on PYTHONPATH below.")

# Notebook-specific plain packages Kaggle's image lacks (e.g. stanza). Installed
# explicitly so they land even if the editable-extra resolve was skipped.
if PIP_PACKAGES.strip():
    _run(f"pip install -q {PIP_PACKAGES}")

# Make `drc` importable BOTH here and in the subprocesses the pipeline spawns.
# The pipeline runs each stage as `python -m drc...`, a fresh process that
# inherits PYTHONPATH (not this cell's sys.path), so we set both. This is what
# makes the run work even if the editable install above didn't register.
SRC = str(WORK / "src")
if SRC not in sys.path:
    sys.path.insert(0, SRC)
os.environ["PYTHONPATH"] = SRC + os.pathsep + os.environ.get("PYTHONPATH", "")

# Work from the repo root so the config's RELATIVE paths resolve under it.
if WORK.exists():
    os.chdir(WORK)
    print("cwd:", os.getcwd())
else:
    print("=" * 70)
    print(f"[error] {WORK} does not exist — the clone FAILED.")
    print("Almost certainly: Kaggle INTERNET IS OFF for this notebook.")
    print("Fix: Settings (right panel) -> Internet -> ON (needs a phone-verified")
    print("account), then re-run. The pipeline also downloads BabyLM from")
    print("HuggingFace, so internet is required regardless of the clone.")
    print("Alternatively, upload the repo as a Kaggle dataset (Add Input).")
    print("=" * 70)
print("drc importable:", _have_drc())
if not _have_drc():
    print("[error] `drc` not importable — see the message above (likely no internet).")

In [ ]:
# --- Install runtime dependencies Kaggle's image doesn't ship. -----------------
# `stanza` (used by the parse stage) is not on the Kaggle GPU image; torch,
# transformers, datasets, scipy, sklearn, matplotlib already are. This runs every
# time and is idempotent, so the parse stage can always `import stanza`.
import subprocess, sys

PACKAGES = "stanza".split()
if PACKAGES:
    print("Installing:", PACKAGES)
    rc = subprocess.call([sys.executable, "-m", "pip", "install", "-q", *PACKAGES])
    print("pip exit code:", rc)
    for pkg in PACKAGES:
        mod = pkg.split("==")[0].split(">=")[0].replace("-", "_")
        try:
            __import__(mod)
            print(f"  import {mod}: OK")
        except Exception as exc:
            print(f"  import {mod}: FAILED — {exc}")
else:
    print("No extra packages to install.")

In [ ]:
# --- Restore prior artifacts so already-done stages resume as "skipped". -------
# When this notebook is chained after another, the previous notebook's
# /kaggle/working is attached as an input dataset under /kaggle/input/<name>/.
# We copy its data/ models/ results/ into our working repo. Safe to re-run;
# dirs_exist_ok lets it merge over an existing tree. Guarded so a missing input
# (e.g. when you run this notebook standalone) is a no-op, not an error.
import glob, shutil
from pathlib import Path

WORK = Path(os.getcwd())  # set by the setup cell
RESTORE_DIRS = ("data", "models", "results")

# Candidate source roots: a chained notebook-output dataset will contain the
# repo's working tree, either at the repo root or one level down.
sources = []
sources += glob.glob("/kaggle/input/*/EMNLP")
sources += glob.glob("/kaggle/input/*/*")
sources += glob.glob("/kaggle/input/*")

restored = []
for src_root in sources:
    src_root = Path(src_root)
    if src_root.resolve() == WORK.resolve():
        continue
    for sub in RESTORE_DIRS:
        src = src_root / sub
        if src.is_dir():
            try:
                shutil.copytree(src, WORK / sub, dirs_exist_ok=True)
                restored.append(str(src))
            except Exception as exc:  # never let a restore failure stop the run
                print(f"[warn] could not restore {src}: {exc}")

if restored:
    print("Restored prior artifacts from:")
    for r in restored:
        print("  ", r)
else:
    print("No prior artifacts found to restore (fine if this is the first stage).")

In [ ]:
# --- Detect GPUs and pick the sweep mode. --------------------------------------
# Set this True to run SEQUENTIALLY on a single card even when two are present
# (e.g. on a T4 x2 box, to use the faster T4 fp16 path without the parallel
# orchestration). The sweep then trains one model at a time on GPU 0.
FORCE_SINGLE_GPU = False

import os
import subprocess

n_gpus = 0
try:
    out = subprocess.run(
        ["nvidia-smi", "-L"], capture_output=True, text=True, check=False
    )
    print(out.stdout.strip() or "(nvidia-smi returned no GPUs)")
    n_gpus = sum(1 for ln in out.stdout.splitlines() if ln.strip().startswith("GPU "))
except FileNotFoundError:
    print("nvidia-smi not found — assuming no GPU (CPU-only).")

SINGLE_GPU = FORCE_SINGLE_GPU or (n_gpus < 2)
if FORCE_SINGLE_GPU and n_gpus >= 2:
    # Use one card only: pin the env so every training subprocess sees GPU 0.
    os.environ["CUDA_VISIBLE_DEVICES"] = "0"
    print("FORCE_SINGLE_GPU set: using GPU 0 only, running sequentially.")
print(f"\nDetected {n_gpus} GPU(s). SINGLE_GPU = {SINGLE_GPU}")

if SINGLE_GPU:
    print("Single-GPU: the sweep trains one model at a time on GPU 0 (no parallelism).")
else:
    print("Dual-GPU: the sweep runs two training jobs in parallel, one per card.")
if n_gpus == 0:
    print("No GPU: training/eval stages will fail; set Accelerator to GPU T4 x2.")

print(
    "\n[note] Precision is auto-selected by the trainer: bf16 on GPUs that"
    "\n       support it (Ampere+), fp16 on a T4 (with a GradScaler), fp32 on"
    "\n       CPU. You do NOT need to edit configs/base.yaml for the T4."
)

In [ ]:
# --- Run the pipeline. ---------------------------------------------------------
# default_phases() returns the wired stages; run_pipeline() isolates failures,
# skips finished stages, blocks stages with unmet deps, and prints a dashboard.
# It never raises on a stage failure. We also wrap the whole cell so that even a
# setup/import problem prints instead of halting — the status cells below still run.
import traceback
from pathlib import Path

CONFIG_PATH = "configs/base.yaml"          # reused by the cells below
# Run all stages: only=None.
only = None  # None == run every stage

try:
    from drc.pipeline import default_phases, run_pipeline
    from drc.data.download import load_config, resolve_path

    cfg = Path(CONFIG_PATH)
    results_dir = resolve_path(cfg, load_config(cfg)["paths"]["results"])
    status = results_dir / "pipeline_status.json"
    stages = default_phases(cfg, single_gpu=SINGLE_GPU)
    results = run_pipeline(stages, status_path=status, only=only)
    # The dashboard is already printed above by run_pipeline.
except Exception:
    print("[error] the run cell hit an exception; the cells below will still run.")
    traceback.print_exc()

In [ ]:
# --- Inspect what we produced. -------------------------------------------------
# Tolerant of missing files and of being run on its own: a fresh or partial run
# just shows fewer artifacts. Wrapped so it never halts a Run All.
import json
import traceback
from pathlib import Path

CONFIG_PATH = globals().get("CONFIG_PATH", "configs/base.yaml")
try:
    from drc.data.download import load_config, resolve_path

    results_dir = resolve_path(Path(CONFIG_PATH), load_config(CONFIG_PATH)["paths"]["results"])

    status_path = results_dir / "pipeline_status.json"
    if status_path.exists():
        data = json.loads(status_path.read_text())
        print("Pipeline status:")
        for name, r in data.items():
            secs = f"{r.get('seconds', 0):.1f}s" if r.get("seconds") else ""
            detail = f"  {r['detail']}" if r.get("detail") else ""
            print(f"  {r['status']:<8} {name:<18} {secs}{detail}")
    else:
        print(f"No {status_path} yet — has the run cell completed?")

    for d in (results_dir, results_dir / "figures"):
        if d.is_dir():
            items = sorted(x.name for x in d.iterdir())
            print(f"\n{d}/ ({len(items)} items):")
            for it in items:
                print("  ", it)
        else:
            print(f"\n{d}/ does not exist yet.")

    decision = results_dir / "decision.txt"
    if decision.exists():
        print("\n=== decision.txt ===")
        print(decision.read_text())
except Exception:
    print("[error] status cell failed; continuing.")
    traceback.print_exc()